# 02 — Memory: Checkpointer vs Store

The #1 interview topic on LangGraph. Most people blur these two — don't.

| | Checkpointer | Store |
|---|---|---|
| **What** | Snapshots graph state at every step | Persists facts across sessions |
| **Scope** | Per `thread_id` | Global (namespace-scoped) |
| **Lifetime** | Ends with thread | Forever (until deleted) |
| **Analogy** | Transaction log / WAL | User profile DB |
| **Dev backend** | `InMemorySaver` | `InMemoryStore` |
| **Prod backend** | `PostgresSaver` / `RedisSaver` | `PostgresStore` / Pinecone |
| **Enables** | Resume, time-travel, multi-turn | Cross-session preferences, learned facts |

## 1. Short-term memory — Checkpointer

In [ ]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage

llm = ChatOllama(model="llama3.2:3b")
checkpointer = InMemorySaver()

def agent_node(state: MessagesState) -> dict:
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

builder = StateGraph(MessagesState)
builder.add_node("agent", agent_node)
builder.add_edge(START, "agent")
builder.add_edge("agent", END)

graph = builder.compile(checkpointer=checkpointer)

# thread_id = conversation scope
config = {"configurable": {"thread_id": "session-A"}}

r1 = graph.invoke({"messages": [HumanMessage(content="My name is Kanishk and I work on ASIN pipelines")]}, config)
print("Turn 1:", r1["messages"][-1].content)

In [ ]:
# Turn 2 — same thread_id, agent automatically has all prior messages
r2 = graph.invoke({"messages": [HumanMessage(content="What is my name and what do I work on?")]}, config)
print("Turn 2:", r2["messages"][-1].content)

### Inspect state & time-travel

In [ ]:
# Current state snapshot
snapshot = graph.get_state(config)
print(f"Messages in state: {len(snapshot.values['messages'])}")
for m in snapshot.values["messages"]:
    print(f"  [{m.type}] {m.content[:80]}")

In [ ]:
# Full history — every checkpoint step
history = list(graph.get_state_history(config))
print(f"Steps recorded: {len(history)}")
for snap in history:
    n_msgs = len(snap.values.get("messages", []))
    print(f"  step {snap.metadata.get('step')}: {n_msgs} messages")

In [ ]:
# Time-travel — replay from step 0 (before turn 1)
earliest = history[-1]  # history is newest-first
print(f"Replaying from step {earliest.metadata.get('step')}")
# graph.invoke(None, earliest.config)  # uncomment to actually replay

## 2. Long-term memory — Store

Survives across threads. Namespace pattern: `(user_id, category)` — same as S3 prefix.

In [ ]:
import uuid
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()

def agent_with_store(state: MessagesState, config: dict) -> dict:
    user_id = config["configurable"].get("user_id", "default")
    namespace = (user_id, "facts")

    # Read existing memories
    memories = store.search(namespace, query=state["messages"][-1].content)
    mem_text = "\n".join(m.value["fact"] for m in memories) if memories else "none"

    system = f"You are a helpful assistant.\nKnown facts about user:\n{mem_text}"
    response = llm.invoke([{"role": "system", "content": system}] + state["messages"])

    # Save new fact to store
    last_msg = state["messages"][-1].content
    store.put(namespace, str(uuid.uuid4()), {"fact": f"User said: {last_msg}"})

    return {"messages": [response]}

builder2 = StateGraph(MessagesState)
builder2.add_node("agent", agent_with_store)
builder2.add_edge(START, "agent")
builder2.add_edge("agent", END)

graph2 = builder2.compile(checkpointer=InMemorySaver(), store=store)

cfg1 = {"configurable": {"thread_id": "session-1", "user_id": "kanishk"}}
r = graph2.invoke({"messages": [HumanMessage(content="I prefer processing NA region first")]}, cfg1)
print("Session 1:", r["messages"][-1].content)

In [ ]:
# New thread — checkpointer forgets session, but store still has the fact
cfg2 = {"configurable": {"thread_id": "session-2", "user_id": "kanishk"}}
r2 = graph2.invoke({"messages": [HumanMessage(content="What do you know about my preferences?")]}, cfg2)
print("Session 2 (new thread):", r2["messages"][-1].content)

## Interview answer

> *"LangGraph separates memory into two layers. The checkpointer is like a WAL — snapshots graph state at every step for a thread_id, enabling resumable workflows and time-travel debugging. The store is long-term memory that persists across threads — user preferences, learned facts. In prod you'd back the checkpointer with Postgres and the store with a vector DB like Pinecone for semantic retrieval. Same separation I used in RISC — session context vs accumulated knowledge about a product."*